# Nemotron LoRA — train on Kaggle (free 2xT4, 4-bit QLoRA)

**Run HEADLESS:** Save Version -> **Save & Run All (Commit)** so it survives browser disconnects.

**Add Input:** the **competition data**, the model **`nemotron-3-nano-30b-a3b-bf16`** (publisher `metric`). **Accelerator: GPU T4 x2. Internet: On.**

Uses PREBUILT mamba wheels (no slow source compile): we pin **torch 2.7** (cxx11abi=TRUE) and install the matching wheels.

## 1. Code + dependencies (pin torch 2.7 for the prebuilt mamba wheels)

In [ ]:
!git clone -b build/nemotron-pipeline https://github.com/SebAustin/NVIDIA-Nemotron-Model-Reasoning-Challenge repo
%cd repo
!pip install -q torch==2.7.0 "transformers>=4.45,<5" peft trl datasets accelerate bitsandbytes psutil einops hf_transfer

## 2. Install prebuilt mamba_ssm + causal_conv1d (matches torch 2.7, ~3 min)

In [ ]:
!pip install -q --no-deps 'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.2.post1/causal_conv1d-1.6.2.post1+cu12torch2.7cxx11abiTRUE-cp312-cp312-linux_x86_64.whl' 'https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1+cu12torch2.7cxx11abiTRUE-cp312-cp312-linux_x86_64.whl'
!python -c "import causal_conv1d, mamba_ssm; print('mamba OK')"

## 3. Competition data (recursive find from the attached dataset)

In [ ]:
import glob, os, shutil
os.makedirs('data', exist_ok=True)
hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
assert hits, "train.csv not found — Add Input -> the competition"
shutil.copy(hits[0], 'data/train.csv'); print('train.csv <-', hits[0])

## 4. EDA + build the SFT data

In [ ]:
!python scripts/01_eda.py --data-dir data
!python scripts/02_prepare_data.py --data-dir data

## 5. Train (4-bit QLoRA on 2xT4)
Loads the base from the attached model mount (auto-detected, no 60 GB download). NF4 + bf16 compute; a smoke test runs first.

In [ ]:
import os
os.environ['QUANT'] = '4bit'
os.environ['NEMOTRON_MAX_MEMORY_GPU'] = '14GiB'
os.environ['SFT_MAX_SEQ_LENGTH'] = '1024'
os.environ['NUM_EPOCHS'] = '2'
!python scripts/03_train_lora.py --data-path data/train_sft.jsonl --output-dir /kaggle/working/lora_adapter

## 6. Package the submission

In [ ]:
!python scripts/05_package_submission.py --adapter-dir /kaggle/working/lora_adapter --output /kaggle/working/submission.zip